# 多智能体
多智能体系统将复杂的应用程序分解成多个专门的智能体，这些智能体协同工作以解决问题。多智能体架构不再依赖单个智能体处理每个步骤，而是将更小、更专注的智能体组合成一个协调的工作流程。

多智能体系统在以下情况下非常有用：
- 单个代理人拥有过多工具，导致其在选择使用哪些工具方面做出糟糕的决策。
- 上下文或记忆变得太大，单个代理无法有效跟踪。
- 任务需要专业化分工（例如，规划师、研究员、数学专家）。

## 多智能体模式

| 图案 | 工作原理 | 控制流 | 示例用例 |
| :--- | :--- | :--- | :--- |
| 工具调用 | 主管代理会调用其他代理作为工具。＂工具＂代理不直接与用户交互——它们只运行任务并返回结果。 | 集中式：所有路由都通过呼叫代理进行。 | 任务编排，结构化工作流程。 |
| 交接 | 当前代理决定将控制权转移给另一个代理。活动代理发生变更，用户可以继续直接与新代理交互。 | 去中心化：代理人可以更改活跃状态。 | 多领域对话，专家接管。 |

## 选择图案
| 问题 | 工具调用 | 交接 |
| :--- | :--- | :--- |
| 需要对工作流程进行集中控制吗？ |  |  |
| 希望客服人员直接与用户互动吗？ |  |是的 |
| 专家之间进行复杂、类似人类的对话？ | 有限 | 强 |

## 自定义代理上下文
多智能体设计的核心是上下文工程——决定每个智能体可以看到哪些信息。LangChain 为您提供了对以下方面的精细控制：
- 对话或状态的哪些部分会传递给每个代理？
- 为子代理商量身定制的专用提示。
- 中间推理的包含/排除。
- 为每个代理自定义输入/输出格式。

系统的质量很大程度上取决于上下文工程。其目标是确保每个代理都能访问执行任务所需的正确数据，无论它是作为工具还是作为主动代理。

## 工具调用
在工具调用中，一个代理（“控制器”）将其他代理视为工具，并在需要时调用它们。控制器负责协调，而工具代理则执行特定任务并返回结果。
流动：

1.控制器接收输入并决定调用哪个工具（子代理）。
2.工具代理根据控制器的指令运行其任务。
3.工具代理将结果返回给控制器。
4.控制器决定下一步操作或结束操作。

<img src="https://i-blog.csdnimg.cn/direct/416ffac2fd4b4e098a5ff2de7627114f.png" alt="image-20220315164332642" style="zoom:50%;" />

## 执行
以下是一个简单的示例，其中主代理通过工具定义获得对单个子代理的访问权限：



In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


subagent1 = create_agent(model="...", tools=[...])

@tool(
    "subagent1_name",
    description="subagent1_description"
)
def call_subagent1(query: str):
    result = subagent1.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

agent = create_agent(model="...", tools=[call_subagent1])

按照这种模式：
- 当主代理call_subagent1认为任务与子代理的描述相符时，就会调用该子代理。
- 子代理独立运行并返回其结果。
- 主控人员收到结果后继续进行协调。
​


### 在哪里可以进行自定义
您可以从多个方面控制主代理及其子代理之间上下文的传递方式：
- 子代理人名称（"subagent1_name"）：这是主代理人对子代理人的称呼。由于它会影响提示，请谨慎选择。
- 子代理描述（"subagent1_description"）：这是主代理“知道”的关于子代理的信息。它直接影响主代理决定何时调用子代理的方式。
- 子智能体的输入：您可以自定义此输入，以便更好地控制子智能体对任务的理解。在上面的示例中，我们query直接传递了智能体生成的输入。
- 子代理的输出：这是返回给主代理的响应。您可以调整返回的内容，以控制主代理如何解释结果。在上面的示例中，我们返回最终消息文本，但您也可以返回其他状态或元数据。
​

## 控制子代理的输入
控制主代理传递给子代理的输入主要有两个途径：
- 修改提示- 调整主代理的提示或工具元数据（即子代理的名称和描述），以便更好地指导何时以及如何调用子代理。
- 上下文注入– 通过调整工具调用从代理的状态中提取信息，添加在静态提示中不切实际的输入（例如，完整的消息历史记录、先前的结果、任务元数据）。

In [ ]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime

class CustomState(AgentState):
    example_state_key: str

@tool(
    "subagent1_name",
    description="subagent1_description"
)
def call_subagent1(query: str, runtime: ToolRuntime[None, CustomState]):
    # Apply any logic needed to transform the messages into a suitable input
    subagent_input = some_logic(query, runtime.state["messages"])
    result = subagent1.invoke({
        "messages": subagent_input,
        # You could also pass other state keys here as needed.
        # Make sure to define these in both the main and subagent's
        # state schemas.
        "example_state_key": runtime.state["example_state_key"]
    })
    return result["messages"][-1].content

## 控制子代理的输出
两种常见的策略，用于控制主代理人从子代理人那里获得的反馈：
- 修改提示- 完善子代理的提示，以准确指定应返回的内容。
    - 当输出不完整、过于冗长或缺少关键细节时，此功能非常有用。
    - 常见的故障模式是子代理执行了工具调用或推理，但没有将结果包含在最终消息中。提醒它，控制器（和用户）只能看到最终输出，因此所有相关信息都必须包含在其中。
- 自定义输出格式——在将子代理的响应返回给主代理之前，在代码中调整或丰富子代理的响应。
    - 例如：除了最终文本之外，还要将特定的状态键传递回主代理。
    - 这需要将结果包装在Command（或等效结构）中，以便您可以将自定义状态与子代理的响应合并。

In [ ]:
from typing import Annotated
from langchain.agents import AgentState
from langchain.tools import InjectedToolCallId
from langgraph.types import Command


@tool(
    "subagent1_name",
    description="subagent1_description"
)
# We need to pass the `tool_call_id` to the sub agent so it can use it to respond with the tool call result
def call_subagent1(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
# You need to return a `Command` object to include more than just a final tool call
) -> Command:
    result = subagent1.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return Command(update={
        # This is the example state key we are passing back
        "example_state_key": result["example_state_key"],
        "messages": [
            ToolMessage(
                content=result["messages"][-1].content,
                # We need to include the tool call id so it matches up with the right tool call
                tool_call_id=tool_call_id
            )
        ]
    })

## 控制子代理的输出
两种常见的策略，用于控制主代理人从子代理人那里获得的反馈：
- 修改提示- 完善子代理的提示，以准确指定应返回的内容。
    - 当输出不完整、过于冗长或缺少关键细节时，此功能非常有用。
    - 常见的故障模式是子代理执行了工具调用或推理，但没有将结果包含在最终消息中。提醒它，控制器（和用户）只能看到最终输出，因此所有相关信息都必须包含在其中。
- 自定义输出格式——在将子代理的响应返回给主代理之前，在代码中调整或丰富子代理的响应。
    - 例如：除了最终文本之外，还要将特定的状态键传递回主代理。
    - 这需要将结果包装在Command（或等效结构）中，以便您可以将自定义状态与子代理的响应合并。



In [ ]:
from typing import Annotated
from langchain.agents import AgentState
from langchain.tools import InjectedToolCallId
from langgraph.types import Command


@tool(
    "subagent1_name",
    description="subagent1_description"
)
# We need to pass the `tool_call_id` to the sub agent so it can use it to respond with the tool call result
def call_subagent1(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
# You need to return a `Command` object to include more than just a final tool call
) -> Command:
    result = subagent1.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return Command(update={
        # This is the example state key we are passing back
        "example_state_key": result["example_state_key"],
        "messages": [
            ToolMessage(
                content=result["messages"][-1].content,
                # We need to include the tool call id so it matches up with the right tool call
                tool_call_id=tool_call_id
            )
        ]
    })

## 交接
在切换过程中，代理可以直接相互传递控制权。“活跃”代理发生变化，用户与当前拥有控制权的代理进行交互。
流动：
1. 当前代理决定需要另一个代理的帮助。
2. 它将控制权（和状态）传递给下一个代理。
3. 新代理会直接与用户交互，直到它决定再次交接或完成为止。